# PyNRPF v0.1.0 - Publication Tables CSV Export

This notebook creates three CSV tables for the paper:
1. Dataset train-test split summary (computed from dataset)
2. Model performance summary (loaded from metrics JSON)
3. Implementation result summary (2023, manual/anonymized examples)

In [ ]:
# Environment and imports
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.io import load_yaml, req, ensure_dir, load_parquet

print('Python:', sys.version)
print('CWD:   ', Path.cwd())
print('REPO:  ', REPO_ROOT)

In [ ]:
# Config and paths
CFG_PATH = REPO_ROOT / 'config' / 'run.yaml'
cfg = load_yaml(CFG_PATH)

RUN_TAG = str(req(cfg, 'run.run_tag'))

DATASET_PATH = (REPO_ROOT / str(req(cfg, 'paths.dataset_parquet'))).resolve()
OUTPUT_DIR = (REPO_ROOT / str(req(cfg, 'paths.output_dir'))).resolve()
TABLE_DIR = OUTPUT_DIR / 'publication_tables'
ensure_dir(TABLE_DIR)

METRICS_JSON = OUTPUT_DIR / f'metrics__{RUN_TAG}.json'

COL_SITE = str(req(cfg, 'data.columns.site'))
COL_TS = str(req(cfg, 'data.columns.ts'))
COL_GT = str(req(cfg, 'data.columns.gt'))

TRAIN_START = pd.Timestamp(str(req(cfg, 'split.train_start'))).date()
TRAIN_END = pd.Timestamp(str(req(cfg, 'split.train_end'))).date()
TEST_START = pd.Timestamp(str(req(cfg, 'split.test_start'))).date()
TEST_END = pd.Timestamp(str(req(cfg, 'split.test_end'))).date()

print('Config loaded from:', CFG_PATH)
print('Run tag:', RUN_TAG)
print('Dataset:', DATASET_PATH)
print('Metrics:', METRICS_JSON)
print('Output table dir:', TABLE_DIR)
print(f'Split: train {TRAIN_START}..{TRAIN_END} | test {TEST_START}..{TEST_END}')

In [ ]:
# Load dataset and split
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

if not METRICS_JSON.exists():
    raise FileNotFoundError(f'Metrics JSON not found: {METRICS_JSON}')

df = load_parquet(DATASET_PATH)
df[COL_TS] = pd.to_datetime(df[COL_TS], errors='raise')
df['date'] = df[COL_TS].dt.date

train_mask = (df['date'] >= TRAIN_START) & (df['date'] <= TRAIN_END)
test_mask = (df['date'] >= TEST_START) & (df['date'] <= TEST_END)

df_train = df.loc[train_mask].copy()
df_test = df.loc[test_mask].copy()
n_other = int((~train_mask & ~test_mask).sum())

print(f'Train rows: {len(df_train):,}')
print(f'Test rows:  {len(df_test):,}')
print(f'Other rows: {n_other:,}')

In [ ]:
# Table 1 - Dataset train-test split summary
def build_split_rows(df_split: pd.DataFrame, split_name: str) -> list[dict]:
    interval_total = int(len(df_split))
    interval_need = int((df_split[COL_GT] < 0).sum())
    interval_pct = round((interval_need / interval_total * 100.0) if interval_total else 0.0, 3)

    day_df = (
        df_split.groupby([COL_SITE, 'date'])
        .agg(need_correction=(COL_GT, lambda s: bool((s < 0).any())))
        .reset_index()
    )
    day_total = int(len(day_df))
    day_need = int(day_df['need_correction'].sum())
    day_pct = round((day_need / day_total * 100.0) if day_total else 0.0, 3)

    return [
        {
            'split': split_name,
            'level': 'day',
            'n_total': day_total,
            'n_need_correction': day_need,
            'pct_need_correction': day_pct,
        },
        {
            'split': split_name,
            'level': 'interval',
            'n_total': interval_total,
            'n_need_correction': interval_need,
            'pct_need_correction': interval_pct,
        },
    ]

table1_rows = []
table1_rows.extend(build_split_rows(df_train, 'train'))
table1_rows.extend(build_split_rows(df_test, 'test'))

table1 = pd.DataFrame(table1_rows)
table1 = table1[['split', 'level', 'n_total', 'n_need_correction', 'pct_need_correction']]
table1

In [ ]:
# Table 2 - Model performance summary (from metrics JSON)
with METRICS_JSON.open('r', encoding='utf-8') as f:
    metrics = json.load(f)

def metric_row(method_key: str, model_level: str, interval_scope: str, result_key: str) -> dict:
    section = metrics[method_key][result_key]
    train = section['train']
    test = section['test']
    return {
        'method': method_key,
        'model_level': model_level,
        'interval_scope': interval_scope,
        'train_precision': round(float(train['precision']), 3),
        'train_recall': round(float(train['recall']), 3),
        'train_f1': round(float(train['f1']), 3),
        'test_precision': round(float(test['precision']), 3),
        'test_recall': round(float(test['recall']), 3),
        'test_f1': round(float(test['f1']), 3),
    }

table2 = pd.DataFrame([
    metric_row('m7_threshold', 'day', 'na', 'day'),
    metric_row('m7_threshold', 'interval', 'all_days', 'interval_all_days'),
    metric_row('m7_threshold', 'interval', 'tp_days_only', 'interval_tp_days_only'),
    metric_row('m8_xgb', 'day', 'na', 'day'),
    metric_row('m8_xgb', 'interval', 'all_days', 'interval_all_days'),
    metric_row('m8_xgb', 'interval', 'tp_days_only', 'interval_tp_days_only'),
])

table2 = table2[[
    'method', 'model_level', 'interval_scope',
    'train_precision', 'train_recall', 'train_f1',
    'test_precision', 'test_recall', 'test_f1',
]]
table2

In [ ]:
# Table 3 - Implementation result (2023, manual/anonymized examples)
table3 = pd.DataFrame([
    {
        'site_name': 'network',
        'year': 2023,
        'min_demand_MW_before': 1481.0,
        'min_demand_MW_after': 1447.0,
        'ts_min_before': '',
        'ts_min_after': '',
    },
    {
        'site_name': 'Site_A',
        'year': 2023,
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -8.21,
        'ts_min_before': '25/10/2023 13:00',
        'ts_min_after': '21/11/2023 14:15',
    },
    {
        'site_name': 'Site_B',
        'year': 2023,
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -6.41,
        'ts_min_before': '17/05/2023 8:15',
        'ts_min_after': '5/10/2023 12:30',
    },
    {
        'site_name': 'Site_C',
        'year': 2023,
        'min_demand_MW_before': 0.00,
        'min_demand_MW_after': -5.24,
        'ts_min_before': '30/11/2023 9:30',
        'ts_min_after': '6/03/2023 13:45',
    },
])

table3['min_demand_MW_correction'] = (
    table3['min_demand_MW_before'] - table3['min_demand_MW_after']
).round(2)

table3 = table3[[
    'site_name', 'year',
    'min_demand_MW_before', 'min_demand_MW_after', 'min_demand_MW_correction',
    'ts_min_before', 'ts_min_after',
]]
table3


In [ ]:
# Export CSV files
TABLE1_PATH = TABLE_DIR / 'table1_dataset_split_summary.csv'
TABLE2_PATH = TABLE_DIR / 'table2_model_performance_summary.csv'
TABLE3_PATH = TABLE_DIR / 'table3_implementation_result_ausgrid_2023.csv'

table1.to_csv(TABLE1_PATH, index=False)
table2.to_csv(TABLE2_PATH, index=False)
table3.to_csv(TABLE3_PATH, index=False)

print(f'Wrote: {TABLE1_PATH}')
print(f'Wrote: {TABLE2_PATH}')
print(f'Wrote: {TABLE3_PATH}')

In [ ]:
# Validation checks
assert TABLE1_PATH.exists(), 'Missing table1 CSV'
assert TABLE2_PATH.exists(), 'Missing table2 CSV'
assert TABLE3_PATH.exists(), 'Missing table3 CSV'

assert len(table1) == 4, f'Table 1 row count expected 4, got {len(table1)}'
assert len(table2) == 6, f'Table 2 row count expected 6, got {len(table2)}'
assert len(table3) == 4, f'Table 3 row count expected 4, got {len(table3)}'

req_t1_cols = ['split', 'level', 'n_total', 'n_need_correction', 'pct_need_correction']
assert table1[req_t1_cols].notna().all().all(), 'Table 1 has nulls in required columns'

metric_cols = [
    'train_precision', 'train_recall', 'train_f1',
    'test_precision', 'test_recall', 'test_f1',
]
assert table2[metric_cols].notna().all().all(), 'Table 2 has missing metrics'

assert set(table3['site_name']) == {'network', 'Site_A', 'Site_B', 'Site_C'}, 'Unexpected Table 3 site names'
assert (table3['year'] == 2023).all(), 'Table 3 year must be 2023'

print('Validation passed.')
print('Table 1 rows:', len(table1))
print('Table 2 rows:', len(table2))
print('Table 3 rows:', len(table3))

table1
